In [ ]:
!pip install sacrebleu

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import sacrebleu
import os
from datetime import datetime
from torchmetrics import Perplexity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Directory setup
results_dir = "pruning_results"
os.makedirs(results_dir, exist_ok=True)

# Load model and tokenizer
model_name = "Helsinki-NLP/opus-mt-de-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_dataset("wmt14", "de-en", split="test")

def calculate_sparsity(model):
    """Calculate sparsity for each weight in the model."""
    sparsity_dict = {}
    
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'weight' in name and 'LayerNorm' not in name and 'bias' not in name:
                total_weights = param.numel()
                zero_weights = torch.sum(param == 0).item()
                sparsity = zero_weights / total_weights * 100 if total_weights > 0 else 0
                sparsity_dict[name] = sparsity
    
    return sparsity_dict

def global_magnitude_prune(model, prune_percent):
    """Global pruning using numpy.percentile for threshold calculation"""
    all_weights = []
    
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'weight' in name and 'LayerNorm' not in name and 'bias' not in name:
                weights = param.data.abs().cpu().numpy().ravel()
                all_weights.append(weights)
        
        if not all_weights:
            return model
        
        flat_weights = np.concatenate(all_weights)
        threshold = np.percentile(flat_weights, prune_percent)
        
        # initial_sparsity = calculate_sparsity(model)
        for name, param in model.named_parameters():
            if 'weight' in name and 'LayerNorm' not in name and 'bias' not in name:
                mask = param.data.abs() > torch.tensor(threshold, device=param.device)
                param.data.mul_(mask)
        
        # final_sparsity = calculate_sparsity(model)
        # print(f"Requested: {prune_percent}% | Achieved: {final_sparsity:.2f}% | "
              # f"New zeros: {final_sparsity - initial_sparsity:.2f}%")
    
    return model

def class_uniform_prune(model, prune_percent):
    """Class-uniform pruning: prune the lowest x% weights within each class/layer"""
    with torch.no_grad():
        # initial_sparsity = calculate_sparsity(model)
        
        for name, param in model.named_parameters():
            if 'weight' in name and 'LayerNorm' not in name and 'bias' not in name:
                weights = param.data.abs().cpu().numpy().ravel()
                if weights.size == 0:
                    continue
                threshold = np.percentile(weights, prune_percent)
                
                mask = param.data.abs() > torch.tensor(threshold, device=param.device)
                param.data.mul_(mask)
        
        # final_sparsity = calculate_sparsity(model)
        # print(f"[Class-Uniform] Requested: {prune_percent}% | Achieved: {final_sparsity:.2f}% | "
              # f"New zeros: {final_sparsity - initial_sparsity:.2f}%")
    
    return model

def class_distribution_prune(model, prune_percent):
    """Class-distribution pruning using σ_c per layer and λ determined to prune x% globally."""
    with torch.no_grad():
        all_std_weight_products = []

        # Step 1: Collect all σ_c for layers and store weight values
        layer_data = []
        for name, param in model.named_parameters():
            if 'weight' in name and 'LayerNorm' not in name and 'bias' not in name:
                weights = param.data.abs().cpu().numpy()
                std_c = np.std(weights)
                if std_c == 0:
                    continue
                scaled_weights = weights / std_c
                all_std_weight_products.extend(scaled_weights.ravel())
                layer_data.append((param, std_c))

        if not all_std_weight_products:
            return model

        # Step 2: Determine λ using global percentile
        lambda_thresh = np.percentile(all_std_weight_products, prune_percent)

        # Step 3: Prune based on class-dependent thresholds
        # initial_sparsity = calculate_sparsity(model)
        for param, std_c in layer_data:
            threshold = lambda_thresh * std_c
            mask = param.data.abs() > torch.tensor(threshold, device=param.device)
            param.data.mul_(mask)

        # final_sparsity = calculate_sparsity(model)
        # print(f"[Class-Distribution] Requested: {prune_percent}% | Achieved: {final_sparsity:.2f}% | "
        #       f"New zeros: {final_sparsity - initial_sparsity:.2f}%")
    
    return model

# Modified evaluate function to calculate perplexity
def evaluate(model, num_samples=1000):
    """Evaluate model on subset of dataset, including perplexity calculation"""
    model.to(device)
    model.eval()
    hypotheses = []
    references = []
    total_loss = 0
    count = 0

    subset = dataset.select(range(num_samples))
    for sample in subset:
        inputs = tokenizer(sample["translation"]["en"], return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs)
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            hypotheses.append(decoded)
            references.append(sample["translation"]["de"])

            # Calculate loss for perplexity
            labels = inputs["input_ids"]
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            count += 1
    
    # Compute average loss and perplexity inside the evaluate function
    avg_loss = total_loss / count
    perplexity = torch.exp(torch.tensor(avg_loss))

    # Calculate BLEU score
    bleu_score = sacrebleu.corpus_bleu(hypotheses, [references])

    print(f"Perplexity: {perplexity.item():.2f}")
    print(f"Bleu_score: {bleu_score.score:.2f}")

    return bleu_score, perplexity.item()

# Pruning methods
pruning_methods = {
    "global": global_magnitude_prune,
    # "class_uniform": class_uniform_prune,
    # "class_distribution": class_distribution_prune
}

# Main loop for pruning and evaluation
for method_name, pruning_fn in pruning_methods.items():
    print(f"\n=== {method_name.replace('_', ' ').title()} Pruning ===")
    
    # Files to store results
    bleu_results = []
    sparsity_results = []
    perplexity_results = []
    
    for prune_pct in range(10, 101, 10):
        print(f"\n--- Pruning {prune_pct}% ---")
        
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
        
        # Apply pruning
        model = pruning_fn(model, prune_pct)

        # Calculate BLEU score and perplexity
        bleu_score, perplexity = evaluate(model)
        bleu_results.append(f"PrunePercent: {prune_pct}, BLEU: {bleu_score.score/100:.2f}")
        perplexity_results.append(f"PrunePercent: {prune_pct}, Perplexity: {perplexity:.2f}")
        
        # Calculate sparsity for each weight in the model
        sparsity_dict = calculate_sparsity(model)
        layer_sparsity = [f"{name}: Sparsity: {sparsity:.2f}%" for name, sparsity in sparsity_dict.items()]
        
        sparsity_results.append(f"PrunePercent: {prune_pct}, " + ", ".join(layer_sparsity))
    
    # Write results to files
    with open(f"{results_dir}/{method_name}_bleu_scores.txt", "w") as f:
        f.write("\n".join(bleu_results))
    
    with open(f"{results_dir}/{method_name}_sparsity.txt", "w") as f:
        f.write("\n".join(sparsity_results))
    
    with open(f"{results_dir}/{method_name}_perplexity.txt", "w") as f:
        f.write("\n".join(perplexity_results))

    print(f"Results saved to {results_dir}/{method_name}_bleu_scores.txt")
    print(f"Results saved to {results_dir}/{method_name}_sparsity.txt")
    print(f"Results saved to {results_dir}/{method_name}_perplexity.txt")
